# Munich rents: iterative specification changes

This study follows a sequence of specification changes on the Munich rent
survey. Each diagnostic motivates a proposed change, which is scored on shared
folds before it is retained; one proposal worsens validation error and is
rejected.

The survey holds 3082 apartments: net rent, floor area, construction year, a
location grade and three fittings indicators. The question is a rent index —
what an apartment with given characteristics should cost — so the model has to
be right about each characteristic separately, not only right on average.

The workflow uses five library functions: `term_report` measures what each term
contributes, `sensitivity` summarizes the fitted effects, `screen_interactions`
screens pairs of predictors, `suggest_specifications` lists candidate changes,
and `compare_specifications` scores each revised model before it is accepted.


:::{admonition} What this study estimates
:class: dropdown

| | |
|---|---|
| **Target** | the sequence of specification changes, each scored before it is kept |
| **Observed response** | `rent`, net monthly rent |
| **Estimand** | the conditional mean rent, revised as the specification changes |
| **Units** | euros per month |
| **Sampling unit** | one apartment |
| **Dependence** | clustered by district |
| **Split / resampling unit** | the apartment |
| **Tuning** | `mstop` inspected at each revision rather than fixed once |
| **Uncertainty** | held-out risk compared between revisions, including the change that made the model worse |
| **Claim type** | descriptive and exploratory; the folds guide successive revisions, so their final risk is optimistic as an estimate of new-data performance |
:::


In [1]:
import numpy as np
import polars as pl

import mboostrs as mb

survey = mb.datasets.munich_rent(output="records")
data = {k: np.asarray(v, dtype=float) for k, v in survey.items()
        if k not in ("district", "rentsqm")}
predictors = {k: v for k, v in data.items() if k != "rent"}
print(f"{len(data['rent'])} apartments, {len(predictors)} predictors")

3082 apartments, 6 predictors


`rentsqm` is excluded because it equals rent divided by area and therefore
contains the response. Making it available would let the model use information
that would be unknown when predicting a new rent.

## Initial specification: `suggest_terms`

`suggest_terms` reads the columns and picks a learner for each — a spline for
the continuous ones, a straight line for the indicators.

In [2]:
first = mb.suggest_terms(data, response="rent")
print(f"rent ~ {first}")

rent ~ bbs(area, df=4.0) + bbs(yearc, df=4.0) + bols(location) + bols(bath) + bols(kitchen) + bols(cheating)


## Term contributions: `term_report`

`term_report` removes each term in turn, refits on shared folds and reports
the change in out-of-fold error: the mean squared error on each held-out fold,
averaged over the five folds. It also compares linear, spline and tree
representations of each continuous predictor while retaining the other terms.
The removal comparison measures the term's incremental predictive contribution;
the shape comparison tests alternatives to its current representation.

In [3]:
report = mb.term_report(f"rent ~ {first}", data, folds=5, mstop=500)
print(f"out-of-fold MSE: {report.baseline:,.0f}\n")
display(pl.DataFrame(report.contributions)
        .select("term", "earns").with_columns(pl.col("earns").round(1)))

out-of-fold MSE: 19,095



term,earns
str,f64
"""bbs(area, df=4.0)""",11084.9
"""bbs(yearc, df=4.0)""",2326.5
"""bols(cheating)""",1331.6
"""bols(location)""",722.0
"""bols(kitchen)""",221.9
"""bols(bath)""",89.0


Removing floor area produces the largest error increase, followed by
construction year and central heating. Removing bath or kitchen raises mean
squared error by about 100 or 200 against a baseline of 19,000. All removal
comparisons have positive point estimates. For bath, `suggest_specifications`
reports a fold standard error of about 110, so its contribution of 89 cannot be
distinguished from zero on these folds.

The shape table compares alternative representations of each predictor.

In [4]:
display(pl.DataFrame(report.shapes)
        .select("term", "current", "best", "as bols", "as bbs", "as btree",
                "gain over current")
        .with_columns(pl.col(pl.Float64).round(1)))

term,current,best,as bols,as bbs,as btree,gain over current
str,str,str,f64,f64,f64,f64
"""bbs(area, df=4.0)""","""bbs""","""bols""",19008.2,19095.1,19451.8,86.9
"""bbs(yearc, df=4.0)""","""bbs""","""bbs""",19990.2,19095.1,19122.8,0.0
"""bols(location)""","""bols""","""bols""",19095.1,19104.8,19105.0,0.0


Replacing the area spline with a straight line lowers the point estimate of
out-of-fold error by 87, less than half a percent of the baseline. The paired
fold standard error of that change is 43 (column `change se` in the comparison
under "Applying the suggested changes"), so the change is about two standard
errors from zero. The reason to prefer the line is simpler interpretation at
comparable predictive error. Construction year is different:
replacing its curve with a line raises error by 895. Location is already linear and
stays that way.

## Fitted effects: `sensitivity`

`sensitivity` evaluates each fitted partial effect across the variable's
range while holding the other predictors at their means.

In [5]:
display(pl.DataFrame(mb.sensitivity(
    mb.gamboost(f"rent ~ {first}", data=data, family=mb.Gaussian(),
                control=mb.boost_control(500, 0.1)),
    predictors)).with_columns(pl.col(pl.Float64).round(2)))

variable,range of the effect,"slope, 10% to 90%",turns
str,f64,f64,bool
"""area""",722.81,4.89,false
"""yearc""",192.62,1.9,true
"""cheating""",126.77,126.77,false
"""location""",96.96,48.48,false
"""kitchen""",73.56,NaN,false
"""bath""",46.89,NaN,false


Monthly rent rises by about 4.9 EUR for each additional square metre of floor
area between the 10th and 90th percentiles of area, and the area effect spans
723 EUR across the whole range. Central heating is worth 127 EUR per month and
the location grade 48 EUR per step. Construction year is the only effect that
**turns**: over the construction-year range it falls and then rises. That
column is the reason the year term needs a spline and the area term does not,
and it is visible without plotting the effects.

## Interaction screening: `screen_interactions`

The initial terms are additive. `screen_interactions` bins each pair of columns
into quantile cells on half the rows, takes the cell means of the residual
there, and measures the mean squared error those means remove on the other half.
A pair that fits only noise removes nothing out of sample.

In [6]:
fitted = mb.gamboost(f"rent ~ {first}", data=data, family=mb.Gaussian(),
                     control=mb.boost_control(500, 0.1))
residual = data["rent"] - np.asarray(fitted.predict()).ravel()
screened = mb.screen_interactions(predictors, residual=residual)
display(pl.DataFrame([{"pair": f"{s.left} x {s.right}",
                       "MSE removed out of sample": round(s.removed, 1)}
                      for s in screened[:5]]))

pair,MSE removed out of sample
str,f64
"""yearc x cheating""",13.6
"""area x cheating""",-4.0
"""location x bath""",-22.6
"""area x yearc""",-32.1
"""area x kitchen""",-52.8


Construction year × central heating is the only pair with a positive screening
score, 13.6, under 0.1% of the baseline error. Area × kitchen scores −52.8, so
the screen does not support it. The comparisons below nevertheless examine area
× kitchen, a kitchen effect that varies with floor area, against the additive
specification; the construction year × central heating pair is not refitted on
this page.

## Suggested changes: `suggest_specifications`

In [7]:
for suggestion in mb.suggest_specifications(report, screened=screened[:5]):
    print(suggestion)

refit bbs(area, df=4.0): +86.89 [measured] (as bols instead of bbs: one slope, so the effect is assumed straight)
drop bols(bath): -89.04 [measured] (what it earns, 89, is less than 2 standard errors (1.1e+02) above zero, so on these folds it cannot be told from a term that earns nothing)
add yearc x cheating: +13.57 [detected] (the pair explains held-out residual variation that neither main effect does. A tensor surface fits a smooth interaction; btree fits axis-aligned thresholds and costs no dense design)


The first two suggestions are based on refits: representing area linearly
reduced out-of-fold MSE by 87, and dropping bath raises it by 89, which is less
than two fold standard errors. The third comes from the residual screen, and no
interaction learner for that pair has been fitted. The tag after each gain
states its basis: `measured` means the change was fitted and scored on the
held-out folds; `detected` means structure was found in the residuals without
any model of it being fitted. Measured suggestions are listed first regardless
of the screening score's magnitude.

## Applying the suggested changes

In [8]:
linear_area = first.replace("bbs(area, df=4.0)", "bols(area)")
tree_pair = mb.expand_pairs(predictors, pairs=[("area", "kitchen")],
                            kind="tree", maxdepth=3)[0]

display(pl.DataFrame(mb.compare_specifications({
    "as suggested by suggest_terms": first,
    "area as a straight line": linear_area,
    "and the area x kitchen tree": linear_area + " + " + tree_pair,
}, data, "rent", folds=5, mstop=500)).select("specification", "out-of-fold risk", "change", "change se").with_columns(pl.col(pl.Float64).round(1)))

specification,out-of-fold risk,change,change se
str,f64,f64,f64
"""as suggested by suggest_terms""",19095.1,0.0,0.0
"""area as a straight line""",19008.2,-86.9,42.8
"""and the area x kitchen tree""",20218.3,1123.2,242.7


The linear area change lowers out-of-fold MSE by 87. Adding the depth-3
area × kitchen tree to it leaves the model 1,123 MSE units above the starting
specification (`suggest_terms`). The screening score evaluates binned residual
means, so it cannot establish that a depth-3 tree would improve this model. The
next comparison tries several learner forms for the same pair.

In [9]:
display(pl.DataFrame(mb.compare_specifications({
    "no interaction term": linear_area,
    "bols(area, by=kitchen)": linear_area + " + bols(area, by=kitchen)",
    "bbs(area, by=kitchen, df=4)": linear_area + " + bbs(area, by=kitchen, df=4)",
    "btree(area, kitchen, maxdepth=1)": linear_area + " + btree(area, kitchen, maxdepth=1)",
    "btree(area, kitchen, maxdepth=3)": linear_area + " + btree(area, kitchen, maxdepth=3)",
}, data, "rent", folds=5, mstop=500)).select("specification", "out-of-fold risk", "change", "change se").with_columns(pl.col(pl.Float64).round(1)))

specification,out-of-fold risk,change,change se
str,f64,f64,f64
"""no interaction term""",19008.2,0.0,0.0
"""bols(area, by=kitchen)""",18853.9,-154.3,109.5
"""bbs(area, by=kitchen, df=4)""",18892.0,-116.2,77.8
"""btree(area, kitchen, maxdepth=…",19317.7,309.5,67.0
"""btree(area, kitchen, maxdepth=…",20218.3,1210.1,220.1


The varying-coefficient term, a linear area effect whose slope depends on the
kitchen indicator, reduces fold MSE by 154 with a paired standard error of
110, about 1.4 standard errors. Every tested tree form increases error, by 310
for the shallowest and 1,210 for the deepest. The refit comparison therefore
favors the varying coefficient on these folds, without separating it clearly
from the specification without the interaction. Screening identified
a pair to investigate; the subsequent comparison identified a useful learner
form without establishing a population interaction.

## Final specification

In [10]:
final = linear_area + " + bols(area, by=kitchen)"
print(f"rent ~ {final}\n")
display(pl.DataFrame(mb.compare_specifications({
    "starting specification": first,
    "final specification": final,
}, data, "rent", folds=5, mstop=500)).select("specification", "out-of-fold risk", "change", "change se").with_columns(pl.col(pl.Float64).round(1)))

closing = mb.term_report(f"rent ~ {final}", data, folds=5, mstop=500)
print("\nremaining suggestions:")
for suggestion in mb.suggest_specifications(closing, top=3):
    print(" ", suggestion)

rent ~ bols(area) + bbs(yearc, df=4.0) + bols(location) + bols(bath) + bols(kitchen) + bols(cheating) + bols(area, by=kitchen)



specification,out-of-fold risk,change,change se
str,f64,f64,f64
"""starting specification""",19095.1,0.0,0.0
"""final specification""",18853.9,-241.2,147.7



remaining suggestions:
  drop bols(kitchen): -0 [measured] (removing it does not raise out-of-fold error)
  drop bols(bath): -71.83 [measured] (what it earns, 71.8, is less than 2 standard errors (1.2e+02) above zero, so on these folds it cannot be told from a term that earns nothing)
  drop bols(area, by=kitchen): -154.3 [measured] (what it earns, 154, is less than 2 standard errors (1.1e+02) above zero, so on these folds it cannot be told from a term that earns nothing)


The final specification's point estimate of out-of-fold error is 241 lower than
the starting specification, with a paired fold standard error of 148, about 1.6
standard errors. The changes replace the area curve with a line, retain the
construction-year curve, and add one varying-coefficient interaction. The
improvement is modest on this fold partition.

With `bols(area, by=kitchen)` in the final model, removing the `kitchen` main
effect does not raise out-of-fold error. The interaction term carries the
kitchen effect instead, as a
term of the form f(area) times kitchen does when the kitchen effect is mostly
a change of slope rather than a change of level. Keeping the main effect
also lets the index report a kitchen premium at fixed area.

## Extension: limitations of the functions used

The workflow uses `term_report` to compare term removal and shapes,
`screen_interactions` to rank residual pairs, `sensitivity` to summarize fitted
effects, and `compare_specifications` to score proposed formulas. The following
limitations affected this workflow.

**Learner comparisons after screening.** The screen scores cell means on a
4x4 grid, while the useful fitted term here was a varying coefficient. This
page compares learner forms by refitting them; the screen does not choose which
form to fit.

**Validation after each change.** Each suggestion is measured relative to
the current specification, so its gain need not persist after another change.
Scoring the cumulative specification after each addition exposes this problem
for the tree.

**Response-derived predictors.** Excluding `rentsqm` required knowing its
definition before fitting. A first fit that attributes a reduction of 1.0 to
one variable and 0.0 to the others would motivate checking for leakage (predictors built from information about the response that is not available when a prediction is made), but
that pattern alone would not establish it.

Every number comes from one five-fold partition of 3,082 rows. Differences of tens of
squared-error units against a baseline near 19,000 are not resolved by these folds: the
area change is under half a percent, and the total point-estimate improvement of 241 is
1.3%. Repeatedly consulting the same folds also makes the final score
optimistic. Three proposed changes are compared here; a longer search needs an
untouched evaluation set.